# Notebook 4: SHAP Analysis & Applicability Domain
**Input:** `data/` arrays from Notebooks 2-3
**Output:** `data/shap_importance.csv`, `data/test_ad_labels.npy`, `figures/shap_importance.png`


In [ ]:
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.DataStructs import ConvertToNumpyArray
from xgboost import XGBClassifier
import shap, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.metrics import matthews_corrcoef

SEED = 42; np.random.seed(SEED)
DATA_DIR = 'data'
os.makedirs('figures', exist_ok=True)

X_train_fp   = np.load(os.path.join(DATA_DIR,'X_train_ecfp4.npy'))
X_test_fp    = np.load(os.path.join(DATA_DIR,'X_test_ecfp4.npy'))
y_train      = np.load(os.path.join(DATA_DIR,'y_train.npy'))
y_test       = np.load(os.path.join(DATA_DIR,'y_test.npy'))
X_train_desc = pd.read_csv(os.path.join(DATA_DIR,'X_train_desc.csv'))
X_test_desc  = pd.read_csv(os.path.join(DATA_DIR,'X_test_desc.csv'))
X_train_comb = np.hstack([X_train_fp, X_train_desc.values])
X_test_comb  = np.hstack([X_test_fp,  X_test_desc.values])
desc_names   = list(X_train_desc.columns)
feat_names   = [f'ECFP4_{i}' for i in range(2048)] + desc_names
df_train     = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))

scale_pos = (y_train==0).sum()/y_train.sum()
model = XGBClassifier(n_estimators=500,learning_rate=0.05,max_depth=6,
    subsample=0.8,colsample_bytree=0.8,scale_pos_weight=scale_pos,
    eval_metric='logloss',random_state=SEED,n_jobs=-1)
model.fit(X_train_comb, y_train)
print('XGB_Combined retrained.')

In [ ]:
# SHAP
explainer    = shap.TreeExplainer(model)
shap_values  = explainer.shap_values(X_test_comb)
mean_abs     = np.abs(shap_values).mean(axis=0)
top20_idx    = np.argsort(mean_abs)[::-1][:20]
top20_names  = [feat_names[i] for i in top20_idx]
top20_vals   = mean_abs[top20_idx]
fp_contrib   = mean_abs[:2048].sum()/mean_abs.sum()*100
desc_contrib = mean_abs[2048:].sum()/mean_abs.sum()*100
print(f'ECFP4 contribution: {fp_contrib:.1f}% | Descriptor: {desc_contrib:.1f}%')
print('\nTop 20 features:')
for i,(n,v) in enumerate(zip(top20_names,top20_vals)): print(f'  {i+1:2d}. {n:<25} {v:.4f}')
shap_df = pd.DataFrame({'feature':feat_names,'mean_abs_shap':mean_abs}).sort_values('mean_abs_shap',ascending=False)
shap_df.to_csv(os.path.join(DATA_DIR,'shap_importance.csv'),index=False)

In [ ]:
# SHAP bar plots
short = [f'FP_bit_{n.split("_")[1]}' if n.startswith('ECFP4_') else n for n in top20_names]
colors = ['#e74c3c' if n.startswith('ECFP4_') else '#3498db' for n in top20_names]
desc_shap  = shap_values[:,2048:]
desc_mean  = np.abs(desc_shap).mean(axis=0)
top15_d    = np.argsort(desc_mean)[::-1][:15]
fig,axes   = plt.subplots(1,2,figsize=(16,7))
axes[0].barh(range(19,-1,-1),top20_vals,color=colors,alpha=0.85,edgecolor='white')
axes[0].set_yticks(range(19,-1,-1)); axes[0].set_yticklabels(short,fontsize=8)
axes[0].set_xlabel('Mean |SHAP|'); axes[0].set_title('Top 20 Features - XGB_Combined')
axes[0].legend(handles=[mpatches.Patch(color='#e74c3c',label='ECFP4 bit'),
                        mpatches.Patch(color='#3498db',label='RDKit descriptor')],fontsize=8)
axes[1].barh(range(14,-1,-1),desc_mean[top15_d],color='#3498db',alpha=0.85,edgecolor='white')
axes[1].set_yticks(range(14,-1,-1)); axes[1].set_yticklabels([desc_names[i] for i in top15_d],fontsize=9)
axes[1].set_xlabel('Mean |SHAP|'); axes[1].set_title('Top 15 Descriptors')
plt.tight_layout()
plt.savefig('figures/shap_importance.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# Applicability domain
def get_fp(smi,radius=2,nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(mol,radius,nBits=nbits)

K = 5
train_fps   = [get_fp(s) for s in df_train['std_smiles']]
train_fps   = [f for f in train_fps if f is not None]
sample_fps  = train_fps[:500]
self_sims   = []
for i,fp in enumerate(sample_fps):
    others = sample_fps[:i]+sample_fps[i+1:]
    sims   = sorted(DataStructs.BulkTanimotoSimilarity(fp,others),reverse=True)
    self_sims.append(np.mean(sims[:K]))
self_sims    = np.array(self_sims)
AD_THRESHOLD = self_sims.mean()-0.5*self_sims.std()
print(f'AD threshold: {AD_THRESHOLD:.4f}')

df_test_data = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
test_fps     = [get_fp(s) for s in df_test_data['std_smiles']]
ad_labels    = []
for fp in test_fps:
    if fp is None: ad_labels.append(0); continue
    sims = sorted(DataStructs.BulkTanimotoSimilarity(fp,train_fps),reverse=True)
    ad_labels.append(1 if np.mean(sims[:K])>=AD_THRESHOLD else 0)
ad_labels = np.array(ad_labels)
np.save(os.path.join(DATA_DIR,'test_ad_labels.npy'),ad_labels)

preds    = model.predict(X_test_comb)
in_mask  = ad_labels==1; out_mask = ad_labels==0
mcc_in   = matthews_corrcoef(y_test[in_mask],preds[in_mask])
mcc_out  = matthews_corrcoef(y_test[out_mask],preds[out_mask])
print(f'Within AD ({in_mask.sum()}, {in_mask.mean()*100:.1f}%): MCC={mcc_in:.3f}')
print(f'Outside AD ({out_mask.sum()}, {out_mask.mean()*100:.1f}%): MCC={mcc_out:.3f}')